In [1]:
import pandas as pd
df_factor = pd.read_parquet('final_feature_matrix_polished2.parquet')
df_factor

,TradingDate,Symbol,is_hs300,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,factor_pos_20d,...,factor_turnover_chg,factor_illiquidity,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_log_ret
0,2010-02-03,000001,True,-0.419412,0.613713,0.627402,0.541997,-0.613713,-0.236243,0.939585,...,0.167893,-0.591555,0.319372,-2.734753,-0.452381,-0.795573,0.477897,-0.238663,0.0,-0.016972
1,2010-02-03,000002,True,0.120885,0.268189,0.435331,-0.229261,-0.268189,-0.007128,-0.321460,...,-0.451399,-0.548991,-0.548896,-0.467890,-1.799874,-0.564989,-1.929665,-0.263356,0.0,-0.004242
2,2010-02-03,000009,True,-0.476138,0.544517,-0.632029,0.934315,-0.544517,-0.551340,-0.479064,...,-0.182673,-0.678319,1.264516,0.034123,1.131501,-0.054876,2.464601,-0.984421,0.0,0.095745
3,2010-02-03,000012,True,-1.256229,-1.338249,-1.340817,-0.825749,1.338249,-1.678090,-1.466028,...,0.443794,-0.025056,0.123518,-1.273525,-0.079102,0.248499,-0.857317,-0.397479,0.0,-0.003327
4,2010-02-03,000021,True,-0.355677,0.674963,0.227034,0.355731,-0.674963,-0.187312,-0.187903,...,-0.550184,0.844361,0.585947,-0.304910,0.440726,-0.199462,-0.528501,0.073503,0.0,0.009686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2334364,2024-12-30,688472,False,-0.053164,-0.020868,-0.579479,-0.265367,0.020868,-0.719673,-0.315587,...,-0.883087,-0.893420,0.897657,2.342325,-1.255127,0.252305,-1.113922,-2.094955,0.0,0.030727
2334365,2024-12-30,688506,False,2.798698,0.225661,0.466909,-0.980781,-0.225661,0.860256,-0.013906,...,1.671173,1.730541,0.825895,2.890364,-0.182732,1.066892,0.074647,0.811098,0.0,-0.005098
2334366,2024-12-30,688561,False,-0.840162,-0.387732,0.081418,-0.498613,0.387732,-0.277653,-0.325313,...,0.126780,0.968039,-0.297715,1.837282,-0.080424,-0.924835,-0.424770,0.638796,0.0,-0.051927
2334367,2024-12-30,688599,True,-1.202224,-2.026339,-1.993219,-0.849288,2.026339,-1.182325,-0.719093,...,0.940073,1.582597,-0.771212,2.222776,0.306306,-1.351603,0.245011,2.128963,0.0,-0.036625


In [27]:
import pandas as pd

# 1. 确保日期格式正确并建立月份索引
df_factor['TradingDate'] = pd.to_datetime(df_factor['TradingDate'])
df_factor['YearMonth'] = df_factor['TradingDate'].dt.to_period('M')

# 2. 提取特征 (Features)：取每个股票每月最后一个交易日的因子值
df_monthly_features = df_factor.sort_values('TradingDate').groupby(['Symbol', 'YearMonth']).last().reset_index()

# 3. 计算目标 (Target)：
# 原数据中 target_log_ret 是 T+1 的日收益，因此 1 月份所有日收益之和 = 2 月份的全月收益
monthly_ret_sum = df_factor.groupby(['Symbol', 'YearMonth'])['target_log_ret'].sum().reset_index()

# 关键对齐步骤：将收益所属月份向前移动一个月，使其与“产生该预测”的月末因子匹配
monthly_ret_sum['YearMonth'] = monthly_ret_sum['YearMonth'] - 1
monthly_ret_sum.rename(columns={'target_log_ret': 'target_monthly_ret'}, inplace=True)

# 4. 合并特征与月度收益率
df_factor_monthly = pd.merge(
    df_monthly_features.drop(columns=['target_log_ret']), 
    monthly_ret_sum, 
    on=['Symbol', 'YearMonth'], 
    how='inner'
)

# 5. 按照交易日期（TradingDate）升序排列
df_factor_monthly = df_factor_monthly.sort_values(by=['TradingDate', 'Symbol']).reset_index(drop=True)

# 6. 保存为 Parquet 格式
df_factor_monthly.to_parquet('df_factor_monthly.parquet')

In [28]:
df_factor_monthly

,Symbol,YearMonth,TradingDate,is_hs300,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,...,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_monthly_ret_x,is_month_end,target_monthly_ret_y
0,600779,2010-02,2010-02-12,True,0.690121,0.848414,1.612065,-0.248941,-0.848414,1.676608,...,-0.673930,-1.113377,-0.645188,-0.186050,-0.503809,1.344352,0.0,-0.008841,True,-0.089385
1,600029,2010-02,2010-02-22,True,-0.588621,-0.219344,0.438071,0.139179,0.219344,0.855814,...,-0.040977,-0.508716,0.708936,-0.039847,1.125171,0.568138,0.0,0.158887,True,0.094646
2,000413,2010-02,2010-02-25,False,-1.672973,-1.137782,-1.338224,-1.119934,1.137782,-1.283195,...,-1.526219,0.911112,0.590471,-0.429389,0.052383,-0.549047,0.0,0.049131,True,0.032340
3,000553,2010-02,2010-02-25,False,0.309649,0.176796,-0.991135,3.429432,-0.176796,-1.712070,...,2.455148,-0.231638,2.660082,-0.618859,2.123024,-0.915525,0.0,0.062428,True,0.066126
4,000651,2010-02,2010-02-25,True,-0.119526,0.123277,-0.101755,0.180504,-0.123277,0.331209,...,1.034304,-0.409951,0.578066,-0.506111,0.039669,1.663298,0.0,0.081640,True,0.098067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115653,688472,2024-11,2024-11-29,False,0.942669,2.994179,1.146609,0.034508,-2.994179,2.431883,...,0.348515,2.346973,0.651500,2.662617,-1.095306,0.622793,0.0,-0.157664,True,-0.157346
115654,688506,2024-11,2024-11-29,False,2.290105,2.503464,2.086610,2.415068,-2.503464,1.941901,...,1.661631,2.886909,1.816233,-0.448256,-0.170427,2.464259,0.0,-0.114185,True,-0.199190
115655,688561,2024-11,2024-11-29,False,1.600082,-1.179198,-0.612583,-0.978070,1.179198,-0.078770,...,-0.216474,1.835178,-0.574994,0.132964,1.115176,0.078435,0.0,-0.078899,True,-0.133112
115656,688599,2024-11,2024-11-29,True,0.560486,0.825428,0.879359,0.578973,-0.825428,1.074836,...,0.152875,2.215543,0.423418,0.037307,0.108796,-0.895662,0.0,-0.195804,True,-0.252351
